In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

In [13]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
LR = 0.01
EPOCHS = 500
MODEL_PATH = "./model/model.pt"
torch.manual_seed(0)
np.random.seed(0)

In [4]:
X_train = torch.from_numpy(
    pd.read_csv("./data/x_train.csv").to_numpy().astype(np.float32)
).to(DEVICE)
y_train = torch.from_numpy(
    pd.read_csv("./data/y_train.csv").to_numpy().astype(np.float32)
).to(DEVICE)
X_test = torch.from_numpy(
    pd.read_csv("./data/x_test.csv").to_numpy().astype(np.float32)
).to(DEVICE)
y_test = torch.from_numpy(
    pd.read_csv("./data/y_test.csv").to_numpy().astype(np.float32)
).to(DEVICE)

# Ensure y shapes are (N,1)
if y_train.ndim == 1:
    y_train = y_train.unsqueeze(1)

if y_test.ndim == 1:
    y_test = y_test.unsqueeze(1)

In [5]:
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

In [6]:
class LogisticRegression(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)  # output single logit

    def forward(self, x):
        logits = self.linear(x)
        prob = torch.sigmoid(logits)
        return prob

In [7]:
model = LogisticRegression(X_train.shape[1]).to(DEVICE)

In [9]:
criterion = nn.BCELoss()  # expects probabilities (after sigmoid)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [10]:
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        preds = model(xb)  # probabilities
        loss = criterion(preds, yb)  # yb should be floats 0.0/1.0
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= len(train_loader.dataset)
    # optional: print progress
    if epoch % 100 == 0 or epoch == 1:
        print(f"Epoch {epoch:04d} loss: {epoch_loss:.6f}")

Epoch 0001 loss: 0.617743
Epoch 0100 loss: 0.321076
Epoch 0200 loss: 0.320479
Epoch 0300 loss: 0.320558
Epoch 0400 loss: 0.320775
Epoch 0500 loss: 0.320699


In [11]:
model.eval()
with torch.no_grad():
    train_preds = model(X_train)
    train_loss = criterion(train_preds, y_train).item()
print(f"Final training loss: {train_loss:.6f}")

Final training loss: 0.320216


In [14]:
torch.save(model.state_dict(), MODEL_PATH)

In [15]:
loaded_model = LogisticRegression(X_train.shape[1]).to(DEVICE)
loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
loaded_model.eval()

LogisticRegression(
  (linear): Linear(in_features=3, out_features=1, bias=True)
)

In [17]:
with torch.no_grad():
    y_prob = loaded_model(X_test).cpu().numpy().flatten()  # probabilities
y_pred = np.round(y_prob).astype(int)  # 0 or 1

# flatten y_test to compare
y_test_np = y_test.cpu().numpy().flatten().astype(int)


acc = accuracy_score(y_test_np, y_pred)
prec = precision_score(y_test_np, y_pred, zero_division=0)
rec = recall_score(y_test_np, y_pred, zero_division=0)
cm = confusion_matrix(y_test_np, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print("Confusion matrix:\n", cm)

Accuracy:  0.8125
Precision: 0.7000
Recall:    0.6087
Confusion matrix:
 [[51  6]
 [ 9 14]]
